# NLLB-LID Validation on NusaX-Senti
# Tujuan: Validasi apakah NLLB-LID bisa diandalkan sebagai evaluator otomatis
# untuk mengukur Linguistic Purity Rate pada data sintetis nanti.

In [9]:
import fasttext
import fasttext.FastText
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download
from pathlib import Path
from collections import Counter

# Fix NumPy 2.x compatibility issue with fasttext
# fasttext uses np.array(obj, copy=False) which is deprecated in NumPy 2.x
_original_predict = fasttext.FastText._FastText.predict

def _patched_predict(self, text, k=1, threshold=0.0, on_unicode_error="strict"):
    if isinstance(text, list):
        all_labels, all_probs = [], []
        for t in text:
            l, p = _patched_predict(self, t, k, threshold, on_unicode_error)
            all_labels.append(l)
            all_probs.append(p)
        return all_labels, all_probs
    else:
        pairs = self.f.predict(text, k, threshold, on_unicode_error)
        if pairs:
            probs, labels = zip(*pairs)
        else:
            probs, labels = ([], ())
        return labels, np.asarray(probs)

fasttext.FastText._FastText.predict = _patched_predict

In [10]:
# Download NLLB-LID model dari HuggingFace
model_path = hf_hub_download(
    repo_id="facebook/fasttext-language-identification",
    filename="model.bin"
)
print(f"Model downloaded to: {model_path}")

# Load model
model = fasttext.load_model(model_path)

Model downloaded to: C:\Users\micha\.cache\huggingface\hub\models--facebook--fasttext-language-identification\snapshots\3af127d4124fc58b75666f3594bb5143b9757e78\model.bin


In [32]:
labels, scores = model.predict("Magello ladde' rupa na iyaré'é nanré, macenning toni.", k=3)
labels, scores

(('__label__bug_Latn', '__label__prs_Arab', '__label__jav_Latn'),
 array([9.97188210e-01, 4.69816034e-04, 4.14403243e-04]))

In [ ]:
# Bahasa target penelitian
# NLLB-LID menggunakan format: __label__<iso639-3>_<script>
TARGET_LANGS = {
    "jav": "__label__jav_Latn",  # Javanese
    "sun": "__label__sun_Latn",  # Sundanese
    "ace": "__label__ace_Latn",  # Acehnese
    "bbc": "__label__bbc_Latn",  # Toba Batak - TIDAK ada di NLLB
}

# Bahasa yang NLLB-LID support (untuk evaluasi linguistic purity)
NLLB_SUPPORTED = {"jav", "sun", "ace"}  # bbc tidak di-support

DATA_DIR = Path("../data/nusax_senti")
SPLITS = ["train", "valid", "test"]

In [ ]:
def predict_language(text: str, k: int = 3):
    # Newlines break fasttext predictions
    clean_text = text.replace("\n", " ").strip()
    if not clean_text:
        return [], []
    labels, scores = model.predict(clean_text, k=k)
    return list(labels), list(scores)


def evaluate_nllb_on_nusax(lang_code: str, split: str = "train"):
    expected_label = TARGET_LANGS[lang_code]
    csv_path = DATA_DIR / lang_code / f"{split}.csv"
    df = pd.read_csv(csv_path)
    
    results = []
    for _, row in df.iterrows():
        text = str(row["text"])
        labels, scores = predict_language(text, k=3)
        
        top1_label = labels[0] if labels else ""
        top1_score = scores[0] if scores else 0.0
        is_correct = (top1_label == expected_label)
        
        # Cek apakah bahasa target ada di top-3
        in_top3 = expected_label in labels
        
        results.append({
            "id": row["id"],
            "text": text[:80] + "..." if len(text) > 80 else text,
            "sentiment": row["label"],
            "expected": expected_label,
            "predicted": top1_label,
            "confidence": round(top1_score, 4),
            "correct_top1": is_correct,
            "in_top3": in_top3,
            # Top-3 untuk analisis
            "top3_labels": str(labels),
            "top3_scores": str([round(s, 4) for s in scores]),
        })
    
    return pd.DataFrame(results)

## Validasi NLLB-LID pada Data NusaX Asli

Jalankan NLLB-LID pada **semua split** (train, valid, test) untuk ketiga bahasa target.
Ini untuk membuktikan bahwa NLLB-LID bisa diandalkan sebagai evaluator otomatis.

In [28]:
# Jalankan evaluasi untuk semua bahasa dan semua split
all_results = {}
summary_rows = []

for lang_code in TARGET_LANGS:
    for split in SPLITS:
        key = f"{lang_code}_{split}"
        print(f"Evaluating {lang_code} ({split})...", end=" ")
        
        df_result = evaluate_nllb_on_nusax(lang_code, split)
        all_results[key] = df_result
        
        n_total = len(df_result)
        n_correct_top1 = df_result["correct_top1"].sum()
        n_in_top3 = df_result["in_top3"].sum()
        avg_confidence = df_result["confidence"].mean()
        
        acc_top1 = n_correct_top1 / n_total * 100
        acc_top3 = n_in_top3 / n_total * 100
        
        summary_rows.append({
            "Language": lang_code,
            "Split": split,
            "Total": n_total,
            "Top-1 Correct": n_correct_top1,
            "Top-1 Accuracy (%)": round(acc_top1, 2),
            "Top-3 Accuracy (%)": round(acc_top3, 2),
            "Avg Confidence": round(avg_confidence, 4),
        })
        
        print(f"Top-1: {acc_top1:.1f}% | Top-3: {acc_top3:.1f}% | Avg Conf: {avg_confidence:.4f}")

print("\nDone!")

Evaluating ace (train)... Top-1: 98.0% | Top-3: 99.6% | Avg Conf: 0.9851
Evaluating ace (valid)... Top-1: 99.0% | Top-3: 99.0% | Avg Conf: 0.9939
Evaluating ace (test)... Top-1: 98.5% | Top-3: 99.5% | Avg Conf: 0.9858
Evaluating ban (train)... Top-1: 64.2% | Top-3: 87.4% | Avg Conf: 0.8252
Evaluating ban (valid)... Top-1: 70.0% | Top-3: 89.0% | Avg Conf: 0.8236
Evaluating ban (test)... Top-1: 68.2% | Top-3: 90.2% | Avg Conf: 0.8390
Evaluating bjn (train)... Top-1: 95.6% | Top-3: 98.2% | Avg Conf: 0.9739
Evaluating bjn (valid)... Top-1: 99.0% | Top-3: 100.0% | Avg Conf: 0.9950
Evaluating bjn (test)... Top-1: 96.2% | Top-3: 97.8% | Avg Conf: 0.9806
Evaluating bbc (train)... Top-1: 0.0% | Top-3: 0.0% | Avg Conf: 0.5117
Evaluating bbc (valid)... Top-1: 0.0% | Top-3: 0.0% | Avg Conf: 0.5082
Evaluating bbc (test)... Top-1: 0.0% | Top-3: 0.0% | Avg Conf: 0.5193
Evaluating bug (train)... Top-1: 98.0% | Top-3: 99.4% | Avg Conf: 0.9828
Evaluating bug (valid)... Top-1: 99.0% | Top-3: 99.0% | Avg 

In [29]:
# Tabel ringkasan akurasi NLLB-LID
df_summary = pd.DataFrame(summary_rows)
print("=" * 80)
print("NLLB-LID VALIDATION SUMMARY ON NusaX-Senti (Native Speaker-Validated Data)")
print("=" * 80)
display(df_summary)

# Rata-rata per bahasa (gabungan semua split)
print("\n--- Rata-rata per Bahasa (semua split) ---")
avg_per_lang = df_summary.groupby("Language").agg({
    "Total": "sum",
    "Top-1 Correct": "sum",
    "Top-1 Accuracy (%)": "mean",
    "Top-3 Accuracy (%)": "mean",
    "Avg Confidence": "mean",
}).round(2)
display(avg_per_lang)

NLLB-LID VALIDATION SUMMARY ON NusaX-Senti (Native Speaker-Validated Data)


,Language,Split,Total,Top-1 Correct,Top-1 Accuracy (%),Top-3 Accuracy (%),Avg Confidence
0,ace,train,500,490,98.00,99.60,0.9851
1,ace,valid,100,99,99.00,99.00,0.9939
2,ace,test,400,394,98.50,99.50,0.9858
3,ban,train,500,321,64.20,87.40,0.8252
4,ban,valid,100,70,70.00,89.00,0.8236
5,ban,test,400,273,68.25,90.25,0.8390
6,bjn,train,500,478,95.60,98.20,0.9739
7,bjn,valid,100,99,99.00,100.00,0.9950
8,bjn,test,400,385,96.25,97.75,0.9806
9,bbc,train,500,0,0.00,0.00,0.5117



--- Rata-rata per Bahasa (semua split) ---


,Total,Top-1 Correct,Top-1 Accuracy (%),Top-3 Accuracy (%),Avg Confidence
Language,,,,,
ace,1000,983,98.50,99.37,0.99
ban,1000,664,67.48,88.88,0.83
bbc,1000,0,0.00,0.00,0.51
bjn,1000,962,96.95,98.65,0.98
bug,1000,986,98.75,99.38,0.99
eng,1000,993,99.50,99.65,0.99
ind,1000,949,94.85,99.67,0.91
jav,1000,953,95.30,99.32,0.96
min,1000,270,26.75,56.20,0.68


## Analisis Error: Kalimat yang Salah Diprediksi

Penting untuk melihat kalimat mana yang NLLB-LID salah prediksi.
Ini membantu memahami limitasi tool dan menentukan threshold yang tepat.

In [ ]:
# Analisis error per bahasa (pada train split)
for lang_code in TARGET_LANGS:
    df_result = all_results[f"{lang_code}_train"]
    errors = df_result[~df_result["correct_top1"]]
    
    print(f"\n{'='*80}")
    print(f"ERRORS for {lang_code.upper()} (train) - {len(errors)} misclassified out of {len(df_result)}")
    print(f"{'='*80}")
    
    if len(errors) == 0:
        print("No errors! Perfect accuracy.")
        continue
    
    # Distribusi: bahasa apa yang diprediksi sebagai gantinya?
    pred_dist = Counter(errors["predicted"].values)
    print(f"\nMispredicted as:")
    for label, count in pred_dist.most_common(10):
        lang_name = label.replace("__label__", "")
        print(f"  {lang_name}: {count} ({count/len(errors)*100:.1f}%)")
    
    # Tampilkan beberapa contoh error
    print(f"\nSample errors (max 10):")
    for _, row in errors.head(10).iterrows():
        pred_lang = row["predicted"].replace("__label__", "")
        print(f"  [{row['confidence']:.3f}] Predicted: {pred_lang} | Text: {row['text']}")

## Analisis Confidence Distribution

Visualisasi distribusi confidence score untuk memahami seberapa "yakin" NLLB-LID
dalam mengenali setiap bahasa. Ini membantu menentukan threshold yang tepat untuk evaluasi data sintetis nanti.

In [ ]:
import matplotlib.pyplot as plt

lang_names = {
    "jav": "Javanese", "sun": "Sundanese",
    "ace": "Acehnese", "bbc": "Toba Batak",
}

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for idx, lang_code in enumerate(TARGET_LANGS):
    df_result = all_results[f"{lang_code}_train"]
    ax = axes[idx]
    
    correct = df_result[df_result["correct_top1"]]["confidence"]
    wrong = df_result[~df_result["correct_top1"]]["confidence"]
    
    ax.hist(correct, bins=20, alpha=0.7, label=f"Correct ({len(correct)})", color="green")
    if len(wrong) > 0:
        ax.hist(wrong, bins=20, alpha=0.7, label=f"Wrong ({len(wrong)})", color="red")
    
    supported = "NLLB supported" if lang_code in NLLB_SUPPORTED else "NOT in NLLB"
    ax.set_title(f"{lang_names[lang_code]} ({lang_code})\n{supported}")
    ax.set_xlabel("Confidence")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)
    ax.axvline(x=0.6, color="orange", linestyle="--", alpha=0.5)

plt.suptitle("NLLB-LID Confidence Distribution on NusaX-Senti (train)", fontsize=14)
plt.tight_layout()
plt.show()

## Kesimpulan

**Bahasa target: Jawa, Sunda, Aceh, Batak Toba**

Hasil validasi NLLB-LID pada data NusaX-Senti asli:

| Bahasa | Top-1 Acc | NLLB-LID Support | Evaluasi Linguistic Purity |
|---|---|---|---|
| Jawa (jav) | ~95% | Ya | NLLB-LID |
| Sunda (sun) | ~93% | Ya | NLLB-LID |
| Aceh (ace) | ~98% | Ya | NLLB-LID |
| Batak Toba (bbc) | 0% | Tidak | Downstream F1 only |

- **jav, sun, ace**: NLLB-LID reliable sebagai evaluator otomatis linguistic purity
- **bbc**: Batak Toba TIDAK ada dalam 218 bahasa NLLB → evaluasi linguistic purity
  tidak dimungkinkan secara otomatis → kualitas data sintetis dinilai via downstream F1-Score